In [3]:
# Connect 4 AI
# Developed by Simran Kajaniya
# Date: March 17, 2025

# This program implements a simple AI for the Connect 4 game using the Minimax algorithm with Alpha-Beta Pruning.
# The game allows a human player to play against the AI on a 6-row by 5-column board.
# The AI evaluates board states and makes optimal moves to maximize its chances of winning.

import numpy as np
import random
import math

ROWS = 6
COLS = 5
PLAYER = 1
AI = 2

# Create board
def create_board():
    return np.zeros((ROWS, COLS), dtype=int)

# Drop piece into the board
def drop_piece(board, row, col, piece):
    board[row][col] = piece

# Check if column is valid
def is_valid_location(board, col):
    return board[0][col] == 0

# Get next available row in a column
def get_next_open_row(board, col):
    for r in range(ROWS-1, -1, -1):
        if board[r][col] == 0:
            return r

# Print the board
def print_board(board):
    print(np.flip(board, 0))

# Check for winning move
def winning_move(board, piece):
    for r in range(ROWS):
        for c in range(COLS - 3):
            if all(board[r, c+i] == piece for i in range(4)):
                return True
    for r in range(ROWS - 3):
        for c in range(COLS):
            if all(board[r+i, c] == piece for i in range(4)):
                return True
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r+i, c+i] == piece for i in range(4)):
                return True
            if all(board[r+3-i, c+i] == piece for i in range(4)):
                return True
    return False

# Evaluate board heuristic
def evaluate_board(board, piece):
    score = 0
    center_array = [int(i) for i in list(board[:, COLS//2])]
    score += center_array.count(piece) * 3

    for r in range(ROWS):
        row_array = [int(i) for i in list(board[r, :])]
        for c in range(COLS - 3):
            window = row_array[c:c+4]
            score += evaluate_window(window, piece)

    for c in range(COLS):
        col_array = [int(i) for i in list(board[:, c])]
        for r in range(ROWS - 3):
            window = col_array[r:r+4]
            score += evaluate_window(window, piece)

    return score

# Evaluate a window of four cells
def evaluate_window(window, piece):
    score = 0
    opp_piece = PLAYER if piece == AI else AI

    if window.count(piece) == 4:
        score += 100
    elif window.count(piece) == 3 and window.count(0) == 1:
        score += 5
    elif window.count(piece) == 2 and window.count(0) == 2:
        score += 2
    if window.count(opp_piece) == 3 and window.count(0) == 1:
        score -= 4

    return score

# Minimax with Alpha-Beta Pruning
def minimax(board, depth, alpha, beta, maximizing_player):
    valid_locations = [c for c in range(COLS) if is_valid_location(board, c)]
    is_terminal = winning_move(board, PLAYER) or winning_move(board, AI) or len(valid_locations) == 0

    if depth == 0 or is_terminal:
        if winning_move(board, AI):
            return (None, 1000000)
        elif winning_move(board, PLAYER):
            return (None, -1000000)
        else:
            return (None, evaluate_board(board, AI))

    if maximizing_player:
        value = -math.inf
        best_col = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, AI)
            new_score = minimax(temp_board, depth-1, alpha, beta, False)[1]
            if new_score > value:
                value = new_score
                best_col = col
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return best_col, value
    else:
        value = math.inf
        best_col = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, PLAYER)
            new_score = minimax(temp_board, depth-1, alpha, beta, True)[1]
            if new_score < value:
                value = new_score
                best_col = col
            beta = min(beta, value)
            if alpha >= beta:
                break
        return best_col, value

# Play game
def play_game():
    board = create_board()
    game_over = False
    print_board(board)

    while not game_over:
        # Human turn
        col = int(input("Enter column (0-4): "))
        if is_valid_location(board, col):
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, PLAYER)
            if winning_move(board, PLAYER):
                print("You win!")
                game_over = True

        print_board(board)

        # AI turn
        if not game_over:
            col, _ = minimax(board, 4, -math.inf, math.inf, True)
            if is_valid_location(board, col):
                row = get_next_open_row(board, col)
                drop_piece(board, row, col, AI)
                if winning_move(board, AI):
                    print("AI wins!")
                    game_over = True
            print_board(board)

play_game()


[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Enter column (0-4): 1
[[0 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
[[0 1 2 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Enter column (0-4): 1
[[0 1 2 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
[[0 1 2 0 0]
 [0 1 0 0 0]
 [0 2 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Enter column (0-4): 1
[[0 1 2 0 0]
 [0 1 0 0 0]
 [0 2 0 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
[[0 1 2 0 0]
 [0 1 2 0 0]
 [0 2 0 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Enter column (0-4): 1
[[0 1 2 0 0]
 [0 1 2 0 0]
 [0 2 0 0 0]
 [0 1 0 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
[[0 1 2 0 0]
 [0 1 2 0 0]
 [0 2 2 0 0]
 [0 1 0 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
Enter column (0-4): 1
[[0 1 2 0 0]
 [0 1 2 0 0]
 [0 2 2 0 0]
 [0 1 0 0 0]
 [0 1 0 0 0]
 [0 1 0 0 0]]
AI wins!
[[0 1 2 0 0]
 [0 1 2 0 0]
 [0 2 2 0 0]
 [0 1 2 0 0]
 [0 1 0 0 0]
 [0 1 0 0 0]]
